# LVIS processor walkthrough

This notebook reads a bounded subset of a real LVIS-C Level-2 TXT product, applies explicit quality criteria, derives RH metrics and canopy-top elevation, reprojects the native footprints, and adds them to the shared data interface.

Install the required dependencies with `python -m pip install -e '.[lvis,examples]'`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tomo_canopy.processors import LVISProcessor
from tomo_canopy.transforms import reproject_points
from tomo_canopy.types import DataCube, GridSpec, Modality

## Locate the LVIS Level-2 source

The complete TXT product contains all 23 RH percentiles. `bounds` reduces the returned footprints, although the ASCII coordinate columns must still be scanned.

In [ ]:
project_root = next(
    path
    for path in (Path.cwd(), Path.cwd().parent)
    if (path / "pyproject.toml").exists()
)
LVIS_PATH = (
    project_root
    / "data/lvisc/raw/LVISC2_Gabon2023_0520_R2507_051090.TXT"
)
assert LVIS_PATH.is_file(), f"Missing {LVIS_PATH}"

WGS84_BOUNDS = (11.6, -0.2, 11.8, 0.1)
MAX_SHOTS = 50_000

## Open and validate footprints

Only variables needed by this workflow are read. No quality filtering happens implicitly in the provider or processor.

In [ ]:
processor = LVISProcessor()
lvis = processor.open(
    LVIS_PATH,
    variables=[
        "rh", "ground_elevation", "sensitivity",
        "channel_rh", "channel_ground", "channel_top",
    ],
    bounds=WGS84_BOUNDS,
    max_shots=MAX_SHOTS,
)
processor.validate(
    lvis, require_crs=True, require_nonempty=True, check_finite=True
)
print("Footprints loaded:", f"{lvis.sizes['shot']:,}")
print("RH percentiles:", lvis.percentile.values.tolist())
lvis

## Apply explicit quality criteria

These thresholds are example choices, not hidden provider defaults. Review the LVIS product documentation and the needs of your analysis before adopting them.

In [ ]:
quality_lvis = processor.filter_quality(
    lvis,
    minimum_sensitivity=0.95,
    accepted_channel_rh=[1],
    accepted_channel_ground=[1],
    accepted_channel_top=[1],
    require_finite_rh=True,
)
print(
    f"Retained {quality_lvis.sizes['shot']:,} of "
    f"{lvis.sizes['shot']:,} footprints"
)
quality_lvis.attrs["quality_filters"]

## Derive RH50, RH98, and canopy-top elevation

RH metrics are heights relative to the detected ground. The example canopy-top elevation is therefore `ground_elevation + RH98`; it remains distinct from the source product's own top-elevation field.

In [ ]:
derived = processor.add_rh_metric(quality_lvis, percentile=50)
derived = processor.add_rh_metric(derived, percentile=98)
derived = processor.add_canopy_top(derived, percentile=98)
derived[["rh50", "rh98", "canopy_top_elevation"]]

## Plot native LVIS footprints

The points represent LVIS laser footprints, not individual airborne-laser returns. A reproducible sample keeps the figure responsive while the processed dataset retains every selected footprint.

In [ ]:
display_sample = processor.sample_shots(
    derived, count=min(10_000, derived.sizes["shot"]), seed=42
)
fig, ax = plt.subplots(figsize=(9, 7))
artist = ax.scatter(
    display_sample.longitude, display_sample.latitude,
    c=display_sample.rh98, s=4, cmap="viridis", alpha=0.75,
)
fig.colorbar(artist, ax=ax, label="LVIS RH98 [m]")
ax.set(
    xlabel="Longitude", ylabel="Latitude",
    title="LVIS-C Level-2 footprints",
)
ax.set_aspect("equal", adjustable="box")
plt.show()

## Reproject and create a native footprint layer

The selected area crosses the equator, so this example uses World Mercator (`EPSG:3395`) as one consistent demonstration CRS. For site analysis, choose the authoritative projected CRS shared by the other modalities. Longitude and latitude remain available after `x/y` are reprojected.

In [ ]:
TARGET_CRS = "EPSG:3395"
projected = reproject_points(derived, target_crs=TARGET_CRS)
lvis_layer = processor.to_data_layer(projected, name="lvis_footprints")
print(lvis_layer)
print("Native bounds:")
display(processor.bounds(projected))

## Add LVIS to a compatible `DataCube`

Native footprints only need to share the cube CRS; they do not have to align with its regular axes. Rasterizing or aggregating LVIS metrics onto those axes is a later `cubegen` operation.

In [ ]:
projected_bounds = processor.bounds(projected)
minimum = dict(zip(projected_bounds.axis.values, projected_bounds.minimum.values))
maximum = dict(zip(projected_bounds.axis.values, projected_bounds.maximum.values))
grid = GridSpec(
    x=np.linspace(minimum["x"], maximum["x"], 10),
    y=np.linspace(minimum["y"], maximum["y"], 10),
    z=np.array([0.0]),
    crs=TARGET_CRS,
)
cube = DataCube(grid=grid)
cube.add_layer(lvis_layer)
print("Modalities:", sorted(cube.modalities))
print("Has LVIS:", cube.has_modality(Modality.LVIS))